In [ ]:
!pip install -U datasets huggingface_hub fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 23.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1


In [1]:
!pip install -U bitsandbytes>=0.46.1

In [2]:
!pip install trl peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 15.0 MB/s eta 0:00:00


In [3]:
import os

from google.colab import userdata
os.environ['HF_TOKEN']=userdata.get('HF_TOKEN')

In [4]:
from datasets import Dataset, load_dataset

ds = load_dataset("newfacade/LeetCodeDataset")
ds

README.md:   0%|          | 0.00/665 [00:00<?, ?B/s]

LeetCodeDataset-train.jsonl:   0%|          | 0.00/93.6M [00:00<?, ?B/s]

LeetCodeDataset-test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2641 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/228 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['task_id', 'question_id', 'difficulty', 'tags', 'problem_description', 'starter_code', 'estimated_date', 'prompt', 'completion', 'entry_point', 'test', 'input_output', 'query', 'response'],
        num_rows: 2641
    })
    test: Dataset({
        features: ['task_id', 'question_id', 'difficulty', 'tags', 'problem_description', 'starter_code', 'estimated_date', 'prompt', 'completion', 'entry_point', 'test', 'input_output', 'query', 'response'],
        num_rows: 228
    })
})

In [5]:
sample = ds['train'].select(range(20)).to_pandas()
sample.head(2)

,task_id,question_id,difficulty,tags,problem_description,starter_code,estimated_date,prompt,completion,entry_point,test,input_output,query,response
0,two-sum,1,Easy,"[Array, Hash Table]",Given an array of integers nums and an integer...,"class Solution:\n def twoSum(self, nums: Li...",2015-08-07,import random\nimport functools\nimport collec...,"class Solution:\n def twoSum(self, nums: Li...",Solution().twoSum,def check(candidate):\n assert candidate(nu...,"[{'input': 'nums = [3,3], target = 6', 'output...",You are an expert Python programmer. You will ...,"To solve this problem efficiently, we can use ..."
1,add-two-numbers,2,Medium,"[Recursion, Linked List, Math]",You are given two non-empty linked lists repre...,# Definition for singly-linked list.\n# class ...,2015-08-07,import heapq\nimport itertools\nfrom sortedcon...,# Definition for singly-linked list.\n# class ...,Solution().addTwoNumbers,def check(candidate):\n assert is_same_list...,"[{'input': 'l1 = [9,8,7], l2 = [1,2,3]', 'outp...",You are an expert Python programmer. You will ...,```python\n# Definition for singly-linked list...


In [ ]:
print(''.join(sample['completion'][0]))

class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:
        d = {}
        for i, x in enumerate(nums):
            if (y := target - x) in d:
                return [d[y], i]
            d[x] = i



In [ ]:
print(''.join(sample['response'][0]))

To solve this problem efficiently, we can use a hash map (dictionary in Python) to store the numbers we have seen so far and their indices. This allows us to check in constant time whether the complement of the current number (i.e., `target - current_number`) has already been encountered. This approach reduces the time complexity from O(n^2) to O(n).

Here's the implementation:

```python
from typing import List

class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:
        # Create a dictionary to store the numbers and their indices
        num_to_index = {}
        
        # Iterate over the list of numbers
        for index, num in enumerate(nums):
            # Calculate the complement
            complement = target - num
            
            # Check if the complement is already in the dictionary
            if complement in num_to_index:
                # If found, return the indices of the complement and the current number
                return [

In [6]:
cols = ds['train'].column_names
columns_needed_to_dropped=[]
for col in cols:
    if col=='problem_description' or col=='response':
        print(type(col))
        continue
    else:
        columns_needed_to_dropped.append(col)
columns_needed_to_dropped

<class 'str'>
<class 'str'>


['task_id',
 'question_id',
 'difficulty',
 'tags',
 'starter_code',
 'estimated_date',
 'prompt',
 'completion',
 'entry_point',
 'test',
 'input_output',
 'query']

In [7]:
ds = ds.remove_columns(columns_needed_to_dropped)
ds

DatasetDict({
    train: Dataset({
        features: ['problem_description', 'response'],
        num_rows: 2641
    })
    test: Dataset({
        features: ['problem_description', 'response'],
        num_rows: 228
    })
})

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
import torch

In [10]:
model_id = 'deepseek-ai/deepseek-coder-1.3b-base'

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code = True)

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

In [11]:
print(tokenizer.chat_template)

print(hasattr(tokenizer, "chat_template"))


None
True


In [12]:
CHAT_TEMPLATE = """{%- for message in messages %}
    {%- if message['role'] == 'user' %}
        {{- '<|user|>\n' + message['content'] + '\n' }}
    {%- elif message['role'] == 'assistant' %}
        {{- '<|assistant|>\n' + message['content'] + eos_token }}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|assistant|>\n' }}
{%- endif %}
"""
print(''.join(CHAT_TEMPLATE))

{%- for message in messages %}
    {%- if message['role'] == 'user' %}
        {{- '<|user|>
' + message['content'] + '
' }}
    {%- elif message['role'] == 'assistant' %}
        {{- '<|assistant|>
' + message['content'] + eos_token }}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|assistant|>
' }}
{%- endif %}



In [13]:
tokenizer.chat_template = CHAT_TEMPLATE
print(tokenizer.chat_template)

{%- for message in messages %}
    {%- if message['role'] == 'user' %}
        {{- '<|user|>
' + message['content'] + '
' }}
    {%- elif message['role'] == 'assistant' %}
        {{- '<|assistant|>
' + message['content'] + eos_token }}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|assistant|>
' }}
{%- endif %}



In [15]:
problem_description = ds['train']['problem_description'][0]
problem_response = ds['train']['response'][0]

#checking the Chat Template:
messages=[
    { 'role': 'user', 'content': problem_description },
    {"role":"assistant", "content":problem_response}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(prompt)

<|user|>
Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target.
You may assume that each input would have exactly one solution, and you may not use the same element twice.
You can return the answer in any order.
 
Example 1:

Input: nums = [2,7,11,15], target = 9
Output: [0,1]
Explanation: Because nums[0] + nums[1] == 9, we return [0, 1].

Example 2:

Input: nums = [3,2,4], target = 6
Output: [1,2]

Example 3:

Input: nums = [3,3], target = 6
Output: [0,1]

 
Constraints:

2 <= nums.length <= 104
-109 <= nums[i] <= 109
-109 <= target <= 109
Only one valid answer exists.

 
Follow-up: Can you come up with an algorithm that is less than O(n2) time complexity?
<|assistant|>
To solve this problem efficiently, we can use a hash map (dictionary in Python) to store the numbers we have seen so far and their indices. This allows us to check in constant time whether the complement of the current number (i.e., `target - current_nu

In [16]:
#Setting the pad-token:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id


In [18]:
def format_to_chat_template(example):


  messages = [
      {
          "role":"user",
          "content":example['problem_description'].strip()
      },
      {
          "role":"assistant",
          "content":example['response'].strip()
      }
  ]

  example['text'] = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

  return example

formatted_ds = ds.map(format_to_chat_template)


In [19]:
print(''.join(formatted_ds['train']['text'][0]))

<|user|>
Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target.
You may assume that each input would have exactly one solution, and you may not use the same element twice.
You can return the answer in any order.
 
Example 1:

Input: nums = [2,7,11,15], target = 9
Output: [0,1]
Explanation: Because nums[0] + nums[1] == 9, we return [0, 1].

Example 2:

Input: nums = [3,2,4], target = 6
Output: [1,2]

Example 3:

Input: nums = [3,3], target = 6
Output: [0,1]

 
Constraints:

2 <= nums.length <= 104
-109 <= nums[i] <= 109
-109 <= target <= 109
Only one valid answer exists.

 
Follow-up: Can you come up with an algorithm that is less than O(n2) time complexity?
<|assistant|>
To solve this problem efficiently, we can use a hash map (dictionary in Python) to store the numbers we have seen so far and their indices. This allows us to check in constant time whether the complement of the current number (i.e., `target - current_nu

In [22]:
eval_dataset

Dataset({
    features: ['problem_description', 'response', 'text'],
    num_rows: 228
})

In [23]:
train_dataset = formatted_ds['train'].select(range(1000))
eval_dataset = formatted_ds['test'].select(range(150))

In [24]:
train_dataset = train_dataset.remove_columns(['problem_description', 'response'])
eval_dataset = eval_dataset.remove_columns(['problem_description', 'response'])
print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['text'],
    num_rows: 1000
})
Dataset({
    features: ['text'],
    num_rows: 150
})


In [ ]:
sample = ds['train'].select(range(20)).map(format_to_chat_template)
sample

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset({
    features: ['problem_description', 'response', 'text'],
    num_rows: 20
})

In [ ]:
print(''.join(sample['text'][0]))

<|user|>
Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target.
You may assume that each input would have exactly one solution, and you may not use the same element twice.
You can return the answer in any order.
 
Example 1:

Input: nums = [2,7,11,15], target = 9
Output: [0,1]
Explanation: Because nums[0] + nums[1] == 9, we return [0, 1].

Example 2:

Input: nums = [3,2,4], target = 6
Output: [1,2]

Example 3:

Input: nums = [3,3], target = 6
Output: [0,1]

 
Constraints:

2 <= nums.length <= 104
-109 <= nums[i] <= 109
-109 <= target <= 109
Only one valid answer exists.

 
Follow-up: Can you come up with an algorithm that is less than O(n2) time complexity?
<|assistant|>
To solve this problem efficiently, we can use a hash map (dictionary in Python) to store the numbers we have seen so far and their indices. This allows us to check in constant time whether the complement of the current number (i.e., `target - current_nu

In [ ]:
sample_map = sample.to_pandas()
sample_map.head(2)

,problem_description,response,text
0,Given an array of integers nums and an integer...,"To solve this problem efficiently, we can use ...",<|user|>\nGiven an array of integers nums and ...
1,You are given two non-empty linked lists repre...,```python\n# Definition for singly-linked list...,<|user|>\nYou are given two non-empty linked l...


In [25]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16
)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config = bnb_config,
    trust_remote_code = True,
    device_map = "auto"
)

pytorch_model.bin:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

In [26]:
# Required before adding LoRA to a quantized model
model = prepare_model_for_kbit_training(model)

model.config.use_cache = False         # disable KV cache during training
model.config.pretraining_tp = 1

In [27]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    inference_mode=False,
)

model = get_peft_model(model, lora_config)

# How many params are actually being trained?
model.print_trainable_parameters()
# → trainable params: ~5M  ||  all params: ~1.3B  ||  trainable%: ~0.4%

trainable params: 14,991,360 || all params: 1,361,463,296 || trainable%: 1.1011


In [28]:
# Calculate total training steps first
total_steps = (len(train_dataset) // (2 * 4)) * 3
# formula: (dataset_size // (batch_size * grad_accum)) * epochs

warmup_steps = int(0.05 * total_steps)  # 5% of total steps as integer
print(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}")

Total steps: 375, Warmup steps: 18


In [29]:
# ── 7. Training Arguments ──────────────────────────────────────────────────────
sft_config = SFTConfig(
    output_dir="./deepseek-leetcode-qlora",

    # Training loop
    num_train_epochs=2,
    per_device_train_batch_size=2,        # keep low for 1.3B on T4/L4
    gradient_accumulation_steps=4,        # effective batch = 2 * 4 = 8

    # Optimizer
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=18,
    weight_decay=0.01,
    optim="paged_adamw_8bit",            # memory-efficient optimizer from bitsandbytes

    # Precision
    #fp16=True,
   bf16=True,                           # use bf16 if your GPU supports it (A100/L4)

    # Sequence
    max_length=1024,                 # LeetCode problems + solutions fit in 1024
    dataset_text_field="text",           # column name in our formatted dataset

    # Logging & Saving
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",                    # change to "wandb" if you use W&B
    disable_tqdm=False
)


In [30]:
from tqdm.notebook import tqdm
from tqdm.auto import tqdm


trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,
    processing_class=tokenizer,
)

# ── 9. Train! ──────────────────────────────────────────────────────────────────
trainer.train()

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32014}.


Epoch,Training Loss,Validation Loss
1,0.338256,0.765421
2,0.263098,0.787229


TrainOutput(global_step=250, training_loss=0.32405397987365725, metrics={'train_runtime': 8582.0865, 'train_samples_per_second': 0.233, 'train_steps_per_second': 0.029, 'total_flos': 1.2357025468121088e+16, 'train_loss': 0.32405397987365725})

In [46]:
from huggingface_hub import login
login()

In [37]:
model_id

'deepseek-ai/deepseek-coder-1.3b-base'

In [39]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(model_id)

model = PeftModel.from_pretrained(base_model, "/content/deepseek-leetcode-qlora/checkpoint-250")

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

In [41]:
model = model.merge_and_unload()

In [44]:
model.save_pretrained("deepseek-leetcode-qlora-merged")
tokenizer.save_pretrained("deepseek-leetcode-qlora-merged")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('deepseek-leetcode-qlora-merged/tokenizer_config.json',
 'deepseek-leetcode-qlora-merged/chat_template.jinja',
 'deepseek-leetcode-qlora-merged/tokenizer.json')

In [47]:
model.push_to_hub("gajasankarraja/leetcode-deepseek-merged-1.3b")
tokenizer.push_to_hub("gajasankarraja/leetcode-deepseek-merged-1.3b")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...g65mlby/model.safetensors:   0%|          | 11.0kB / 2.69GB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/gajasankarraja/leetcode-deepseek-merged-1.3b/commit/05cbc7e9252003d5bf4d603885f2457d56548923', commit_message='Upload tokenizer', commit_description='', oid='05cbc7e9252003d5bf4d603885f2457d56548923', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gajasankarraja/leetcode-deepseek-merged-1.3b', endpoint='https://huggingface.co', repo_type='model', repo_id='gajasankarraja/leetcode-deepseek-merged-1.3b'), pr_revision=None, pr_num=None)